# 01 A0 Protocol Audit

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

DRIVE_MOUNT = Path('/content/drive')
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'ECG-Ramba'

def _drive_root_ready(root):
    try:
        return root.is_dir() and any(root.iterdir())
    except Exception:
        return False

try:
    from google.colab import drive
    if not _drive_root_ready(DRIVE_ROOT):
        try:
            drive.mount(str(DRIVE_MOUNT))
        except Exception as exc:
            print(f'Drive mount initial attempt failed or was stale: {exc}')
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
    else:
        print('Drive root already visible:', DRIVE_ROOT)
except Exception as exc:
    print(f'Drive mount skipped or unavailable: {exc}')

if not _drive_root_ready(DRIVE_ROOT):
    raise RuntimeError(
        f'Google Drive root is not readable at {DRIVE_ROOT}. Restart/remount before continuing.'
    )
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path(os.environ.get('ECG_RAMBA_REPO_DIR', '/content/ECG-RAMBA'))
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def run_logged(cmd, log_path, cwd=None, check=True):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    log_path = Path(log_path)
    if not log_path.is_absolute():
        log_path = run_cwd / log_path
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'$ {cmd}', flush=True)
    try:
        log_relative = log_path.resolve().relative_to((REPO_DIR / 'reports' / 'revision').resolve())
    except ValueError:
        log_relative = Path('logs') / log_path.name
    durable_log_path = MIRROR_REVISION_ROOT / log_relative
    durable_log_path.parent.mkdir(parents=True, exist_ok=True)
    from contextlib import ExitStack
    with ExitStack() as stack:
        log = stack.enter_context(log_path.open('a', encoding='utf-8'))
        durable_log = stack.enter_context(durable_log_path.open('a', encoding='utf-8'))
        proc = subprocess.Popen(
            cmd, shell=True, cwd=str(run_cwd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            durable_log.write(line)
            durable_log.flush()
        return_code = proc.wait()
    print('Command log:', log_path)
    print('Durable command log:', durable_log_path)
    if check and return_code:
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
        if os.environ.get('ECG_RAMBA_ALLOW_STALE_REPO', '0') != '1':
            raise RuntimeError('git pull failed; refusing to continue with stale code. Set ECG_RAMBA_ALLOW_STALE_REPO=1 only for debugging.')
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)

# BEGIN FORENSIC CODE AUTHORITY PIN
FORENSIC_CODE_AUTHORITY_CAPABILITY = 'canonical_versioned_git_release_v2'
FORENSIC_CODE_AUTHORITY_SCHEMA_VERSION = 2
_AUTHORITY_BOOTSTRAP_ALLOWED = False
_AUTHORITY_DEFAULT_RELEASE_REF = 'refs/tags/ecg-ramba-revision-20260722-v8'

def _pin_forensic_code_authority():
    import json as _authority_json
    import os as _authority_os
    import re as _authority_re
    import subprocess as _authority_subprocess
    import uuid as _authority_uuid
    from datetime import datetime as _authority_datetime, timezone as _authority_timezone
    from pathlib import Path as _AuthorityPath
    from scripts.revision.artifact_mirror import PublishLock as _AuthorityPublishLock

    manifest_path = _AuthorityPath(MIRROR_REVISION_ROOT) / 'manifests' / 'notebook_code_authority.json'
    requested_commit = _authority_os.environ.get('ECG_RAMBA_AUTHORITY_COMMIT', '').strip().lower()
    requested_release_ref = _authority_os.environ.get(
        'ECG_RAMBA_AUTHORITY_REF', _AUTHORITY_DEFAULT_RELEASE_REF
    ).strip()
    reset_requested = _authority_os.environ.get('ECG_RAMBA_RESET_CODE_AUTHORITY', '0') == '1'
    legacy_rotate_requested = (
        _authority_os.environ.get('ECG_RAMBA_ROTATE_CODE_AUTHORITY_TO_BRANCH_HEAD', '0') == '1'
    )
    if legacy_rotate_requested:
        raise RuntimeError(
            'Implicit authority rotation to a moving branch head is disabled. '
            'Notebook 00 follows only a reviewed versioned release tag. For an emergency explicit '
            'pin, set ECG_RAMBA_RESET_CODE_AUTHORITY=1 with a full ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    rotate_to_branch_head = False
    commit_pattern = _authority_re.compile(r'[0-9a-f]{40}')
    release_ref_pattern = _authority_re.compile(r'refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*')

    def git(*args, check=True):
        result = _authority_subprocess.run(
            ['git', *args],
            cwd=str(REPO_DIR),
            check=False,
            text=True,
            stdout=_authority_subprocess.PIPE,
            stderr=_authority_subprocess.STDOUT,
        )
        if check and result.returncode:
            raise RuntimeError(
                'Code-authority git command failed: git '
                + ' '.join(args)
                + '\n'
                + (result.stdout or '')[-4000:]
            )
        return result

    def resolve_annotated_release_ref(release_ref):
        if not release_ref_pattern.fullmatch(release_ref):
            raise RuntimeError(
                'ECG_RAMBA_AUTHORITY_REF must be a full refs/tags/... name. '
                'Moving branches are not accepted as code authority.'
            )
        object_type = git('cat-file', '-t', release_ref, check=False)
        if object_type.returncode or object_type.stdout.strip() != 'tag':
            raise RuntimeError(
                f'Code-authority release ref is missing or is not an annotated Git tag: {release_ref}. '
                'Run the current Notebook 00 after the reviewed release tag has been pushed.'
            )
        object_id = git('rev-parse', '--verify', release_ref).stdout.strip().lower()
        commit = git('rev-parse', '--verify', release_ref + '^{commit}').stdout.strip().lower()
        if not commit_pattern.fullmatch(object_id) or not commit_pattern.fullmatch(commit):
            raise RuntimeError(f'Could not resolve an immutable object and commit for {release_ref}.')
        return commit, object_id

    if reset_requested and not _AUTHORITY_BOOTSTRAP_ALLOWED:
        raise RuntimeError(
            'Only Notebook 00 may rotate the canonical code authority. '
            'Run Notebook 00 with ECG_RAMBA_RESET_CODE_AUTHORITY=1 and an explicit '
            'ECG_RAMBA_AUTHORITY_COMMIT.'
        )
    if reset_requested and not commit_pattern.fullmatch(requested_commit):
        raise RuntimeError(
            'Authority reset requires ECG_RAMBA_AUTHORITY_COMMIT as a full 40-character git SHA.'
        )

    tracked_status = git('status', '--porcelain', '--untracked-files=no').stdout.strip()
    if tracked_status:
        raise RuntimeError(
            'Tracked files differ from git before authority checkout. Use a fresh Colab clone; '
            'authority pinning will not stash or overwrite local edits.\n' + tracked_status[:4000]
        )

    fetch = git('fetch', 'origin', '--prune', '--tags', check=False)
    if fetch.returncode:
        print('WARNING: git fetch failed; accepting only an already-present pinned commit/release tag.')
        print((fetch.stdout or '')[-2000:])

    manifest = None
    manifest_raw = None
    with _AuthorityPublishLock(
        _AuthorityPath(MIRROR_REVISION_ROOT),
        run_id='authority-read-' + _authority_uuid.uuid4().hex,
    ):
        if manifest_path.is_file():
            manifest_raw = manifest_path.read_text(encoding='utf-8')
            manifest = _authority_json.loads(manifest_raw)

    authority_update_needed = False
    update_reason = None
    release_ref = None
    release_ref_object_id = None

    if reset_requested:
        expected_commit = requested_commit
        authority_update_needed = True
        update_reason = 'explicit_environment_sha'
    elif manifest is None:
        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            raise FileNotFoundError(
                'Canonical code-authority manifest is missing. Run Notebook 00 first in a fresh runtime; '
                'downstream notebooks fail closed instead of following a moving branch.'
            )
        release_ref = requested_release_ref
        expected_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
        authority_update_needed = True
        update_reason = 'initial_versioned_release_bootstrap'
    else:
        manifest_capability = manifest.get('capability')
        manifest_schema = int(manifest.get('schema_version', 0))
        legacy_manifest = (
            manifest_capability == 'canonical_git_commit_pin_v1' and manifest_schema == 1
        )
        current_manifest = (
            manifest_capability == 'canonical_versioned_git_release_v2' and manifest_schema == 2
        )
        if not legacy_manifest and not current_manifest:
            raise RuntimeError('Canonical code-authority manifest capability/schema is invalid.')
        manifest_commit = str(manifest.get('git_commit', '')).strip().lower()
        if not commit_pattern.fullmatch(manifest_commit):
            raise RuntimeError('Canonical code-authority manifest lacks a full git SHA.')
        if str(manifest.get('repository_url', '')).rstrip('/') != str(REPO_URL).rstrip('/'):
            raise RuntimeError('Canonical code-authority repository URL differs from this notebook.')
        if str(manifest.get('branch', '')) != str(BRANCH):
            raise RuntimeError('Canonical code-authority branch differs from this notebook runtime.')

        if not _AUTHORITY_BOOTSTRAP_ALLOWED:
            if legacy_manifest:
                raise RuntimeError(
                    'Canonical code authority uses the legacy schema. Run the current Notebook 00 once '
                    'to migrate it to the reviewed versioned release before running downstream notebooks.'
                )
            expected_commit = manifest_commit
            if requested_commit and requested_commit != expected_commit:
                raise RuntimeError(
                    'ECG_RAMBA_AUTHORITY_COMMIT differs from the canonical authority manifest. '
                    'Rotate authority explicitly in Notebook 00; do not override it downstream.'
                )
        else:
            release_ref = requested_release_ref
            release_commit, release_ref_object_id = resolve_annotated_release_ref(release_ref)
            manifest_ref = str(manifest.get('authority_ref', '')).strip()
            manifest_ref_object = str(manifest.get('authority_ref_object_id', '')).strip().lower()
            if current_manifest and manifest_ref == release_ref:
                if manifest_commit != release_commit or manifest_ref_object != release_ref_object_id:
                    raise RuntimeError(
                        f'Code-authority release tag moved or changed after it was recorded: {release_ref}. '
                        'Publish a new versioned tag instead of retagging an existing release.'
                    )
                expected_commit = manifest_commit
            else:
                expected_commit = release_commit
                authority_update_needed = True
                update_reason = (
                    'legacy_manifest_migration'
                    if legacy_manifest
                    else 'versioned_release_upgrade'
                )

    if not commit_pattern.fullmatch(expected_commit):
        raise RuntimeError('Notebook 00 could not resolve a full code-authority git SHA.')
    git('cat-file', '-e', expected_commit + '^{commit}')
    git('checkout', '--detach', expected_commit)
    observed_commit = git('rev-parse', 'HEAD').stdout.strip().lower()
    if observed_commit != expected_commit:
        raise RuntimeError(
            f'Code-authority checkout mismatch: expected={expected_commit} observed={observed_commit}'
        )

    if authority_update_needed:
        previous_commit = None if manifest is None else str(manifest.get('git_commit', '')).strip().lower()
        previous_ref = None if manifest is None else manifest.get('authority_ref')
        manifest = {
            'capability': 'canonical_versioned_git_release_v2',
            'schema_version': 2,
            'git_commit': expected_commit,
            'repository_url': str(REPO_URL),
            'branch': str(BRANCH),
            'established_utc': _authority_datetime.now(_authority_timezone.utc).isoformat(),
            'established_by': '00_colab_bootstrap.ipynb',
            'selection': (
                'explicit_environment_sha'
                if release_ref is None
                else 'verified_annotated_versioned_release_tag'
            ),
            'authority_ref': release_ref,
            'authority_ref_kind': None if release_ref is None else 'annotated_git_tag',
            'authority_ref_object_id': release_ref_object_id,
            'update_reason': update_reason,
            'previous_git_commit': previous_commit,
            'previous_authority_ref': previous_ref,
        }
        with _AuthorityPublishLock(
            _AuthorityPath(MIRROR_REVISION_ROOT),
            run_id='authority-write-' + _authority_uuid.uuid4().hex,
        ):
            concurrent_raw = manifest_path.read_text(encoding='utf-8') if manifest_path.is_file() else None
            if concurrent_raw != manifest_raw and not reset_requested:
                concurrent = _authority_json.loads(concurrent_raw) if concurrent_raw else None
                if not (
                    concurrent
                    and concurrent.get('capability') == 'canonical_versioned_git_release_v2'
                    and int(concurrent.get('schema_version', 0)) == 2
                    and str(concurrent.get('git_commit', '')).lower() == expected_commit
                    and concurrent.get('authority_ref') == release_ref
                    and str(concurrent.get('authority_ref_object_id', '')).lower()
                    == str(release_ref_object_id or '').lower()
                ):
                    raise RuntimeError('A concurrent runtime established a different code authority.')
                manifest = concurrent
            else:
                manifest_path.parent.mkdir(parents=True, exist_ok=True)
                temporary = manifest_path.with_name(
                    manifest_path.name + '.partial.' + _authority_uuid.uuid4().hex
                )
                with temporary.open('w', encoding='utf-8') as handle:
                    handle.write(_authority_json.dumps(manifest, indent=2, sort_keys=True) + '\n')
                    handle.flush()
                    _authority_os.fsync(handle.fileno())
                _authority_os.replace(temporary, manifest_path)
        print('Established/updated canonical code authority:', manifest_path)

    _authority_os.environ['ECG_RAMBA_AUTHORITY_COMMIT'] = expected_commit
    _authority_os.environ.pop('ECG_RAMBA_RESET_CODE_AUTHORITY', None)
    globals()['CODE_AUTHORITY_MANIFEST_PATH'] = manifest_path
    globals()['CODE_AUTHORITY'] = manifest
    print('Pinned code authority:', expected_commit)
    print('Authority release  :', manifest.get('authority_ref'))
    print('Authority manifest :', manifest_path)
    return manifest

CODE_AUTHORITY = _pin_forensic_code_authority()
# END FORENSIC CODE AUTHORITY PIN

_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

# Route the Notebook 01 legacy helper through the common forensic history wrapper.
run = run_logged

# Forensic run-history wrapper. The legacy helper writes live output while this
# wrapper gives every invocation a unique, durable stage/run_id log and retains
# the requested stable path as the latest-run convenience copy.
FORENSIC_RUN_HISTORY_CAPABILITY = 'stage_run_id_v1'
_forensic_base_run = run

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    import inspect as _forensic_inspect
    import os as _forensic_os
    import shutil as _forensic_shutil
    import subprocess as _forensic_subprocess
    import time as _forensic_time
    import uuid as _forensic_uuid
    from datetime import datetime as _forensic_datetime, timezone as _forensic_timezone
    from pathlib import Path as _ForensicPath

    run_cwd = _ForensicPath(cwd) if cwd else _ForensicPath.cwd()
    if log_path is None:
        stable_log = run_cwd / 'reports' / 'revision' / 'logs' / 'notebook_command_latest.log'
    else:
        stable_log = _ForensicPath(log_path)
        if not stable_log.is_absolute():
            stable_log = run_cwd / stable_log
    stable_log.parent.mkdir(parents=True, exist_ok=True)
    stage = stable_log.stem
    run_id = _forensic_datetime.now(_forensic_timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '-' + _forensic_uuid.uuid4().hex[:10]
    history_log = stable_log.parent / 'history' / stage / f'{run_id}.log'
    history_log.parent.mkdir(parents=True, exist_ok=True)

    canonical_root = globals().get('MIRROR_REVISION_ROOT')
    if canonical_root is None and 'DRIVE_ROOT' in globals():
        canonical_root = _ForensicPath(DRIVE_ROOT) / 'revision_artifacts' / 'reports' / 'revision'
    canonical_history = None
    if canonical_root is not None:
        canonical_root = _ForensicPath(canonical_root)
        canonical_history = canonical_root / 'logs' / 'history' / stage / f'{run_id}.log'
        canonical_history.parent.mkdir(parents=True, exist_ok=True)

    started = _forensic_datetime.now(_forensic_timezone.utc).isoformat()
    header = f'run_id={run_id}\nstage={stage}\nstarted_utc={started}\ncommand={cmd}\n--- output ---\n'
    history_log.write_text(header, encoding='utf-8')
    if canonical_history is not None:
        canonical_history.write_text(header, encoding='utf-8')

    parameters = _forensic_inspect.signature(_forensic_base_run).parameters
    kwargs = {}
    if 'cwd' in parameters:
        kwargs['cwd'] = run_cwd
    if 'check' in parameters:
        kwargs['check'] = check
    if 'tail_lines' in parameters:
        kwargs['tail_lines'] = tail_lines
    if 'log_path' in parameters:
        kwargs['log_path'] = history_log

    return_code = -1
    caught = None
    completed = None
    try:
        completed = _forensic_base_run(cmd, **kwargs)
        return_code = int(getattr(completed, 'returncode', 0))
    except BaseException as exc:
        caught = exc
        return_code = int(getattr(exc, 'returncode', -1))
    finally:
        footer = (
            '\n--- end ---\n'
            f'ended_utc={_forensic_datetime.now(_forensic_timezone.utc).isoformat()}\n'
            f'return_code={return_code}\n'
        )
        with history_log.open('a', encoding='utf-8') as handle:
            handle.write(footer)
            handle.flush()
            _forensic_os.fsync(handle.fileno())
        if canonical_history is not None:
            # The underlying helper streams to this same canonical history path
            # when supported; append the footer or refresh from the local copy.
            try:
                _forensic_shutil.copy2(history_log, canonical_history)
            except Exception as exc:
                print('WARNING: durable history refresh failed:', exc)
        try:
            _forensic_shutil.copy2(history_log, stable_log)
            if canonical_root is not None:
                try:
                    revision_base = (_ForensicPath(globals().get('REPO_DIR', run_cwd)) / 'reports' / 'revision').resolve()
                    relative = stable_log.resolve().relative_to(revision_base)
                except (ValueError, TypeError):
                    relative = _ForensicPath('logs') / stable_log.name
                canonical_stable = canonical_root / relative
                canonical_stable.parent.mkdir(parents=True, exist_ok=True)
                _forensic_shutil.copy2(history_log, canonical_stable)
        except Exception as exc:
            print('WARNING: latest log refresh failed:', exc)
    print('Run history log:', history_log)
    if canonical_history is not None:
        print('Durable run history log:', canonical_history)
    if caught is not None:
        raise caught
    if check and return_code:
        raise _forensic_subprocess.CalledProcessError(return_code, cmd)
    return completed

run_logged = run


## Revision Plan Files

In [ ]:
import pandas as pd

for path in [
    'docs/revision_plan/task_board.csv',
    'docs/revision_plan/claim_evidence_map.csv',
    'docs/revision_plan/experiment_registry.csv',
    'docs/revision_plan/a0_resolution_checklist.csv',
]:
    print('\n' + path)
    display(pd.read_csv(path).head(20))


## Run Protocol Audit

In [ ]:
run_logged(
    'python -u scripts/revision/00_audit_protocol.py',
    'reports/revision/logs/a0_protocol_audit.log',
)


## Inspect Audit Warnings

In [ ]:
audit = json.loads(Path('reports/revision/audit_protocol.json').read_text(encoding='utf-8'))
print('Warnings:')
for warning in audit.get('warnings', []):
    print('-', warning)

print('\nDataset paths:')
for key, info in audit['artifacts'].get('dataset_paths', {}).items():
    print(f"{key}: exists={info['exists']} path={info['path']}")


## Validate A0 Resolution Decisions

In [ ]:
run_logged(
    'python -u scripts/revision/00_a0_resolution_gate.py',
    'reports/revision/logs/a0_resolution_gate.log',
)


## Inspect A0 Completion Gate

In [ ]:
a0_status = json.loads(Path('reports/revision/a0_resolution_status.json').read_text(encoding='utf-8'))
print('A0 status        :', a0_status['status'])
print('Audit complete   :', a0_status['audit_complete'])
print('Protocol ready   :', a0_status['protocol_ready'])
print('Deferred blockers:', a0_status['deferred_blocker_ids'])
print('Meaning:', a0_status['meaning'])
for blocker in a0_status['blockers']:
    print(
        blocker['blocker_id'],
        blocker['resolution_status'],
        'valid=' + str(blocker['valid']),
        blocker['restriction'],
    )


## Inventory Current Artifacts

In [ ]:
run_logged(
    'python -u scripts/revision/05_artifact_inventory.py',
    'reports/revision/logs/a0_artifact_inventory.log',
)
run_logged(
    f'python -u scripts/revision/artifact_mirror.py publish --verify-existing size --mirror-root "{MIRROR_REVISION_ROOT}"',
    'reports/revision/logs/a0_mirror_publish.log',
)
